# Edge tunnel (backup): SSH into this Kaggle session

1. Add-ons -> Secrets: `SSH_PUBKEY` = the contents of your `~/.ssh/id_ed25519.pub` (public key only).
2. Accelerator as needed (GPU T4 x2), Internet ON. **Run All in the editor** (not Save Version), keep the tab open.
3. Copy the `*.trycloudflare.com` hostname printed below, then on the PC:
   `bash scripts/edge/kaggle_tunnel_client.sh <hostname>`
4. Stop the session from the editor when done (it otherwise holds GPU quota until the 12 h limit).

Password login is disabled; only the key in `SSH_PUBKEY` can log in.


In [ ]:
import os, subprocess, pathlib, time, re

from kaggle_secrets import UserSecretsClient
pub = UserSecretsClient().get_secret("SSH_PUBKEY").strip()
if not pub.startswith(("ssh-ed25519", "ssh-rsa", "ecdsa-")):
    raise SystemExit("SSH_PUBKEY does not look like a public key (expected ssh-ed25519 ... / ssh-rsa ...)")

def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if r.returncode:
        raise SystemExit(f"failed: {cmd}\n{r.stderr[-800:]}")
    return r.stdout

sh("apt-get -qq update && DEBIAN_FRONTEND=noninteractive apt-get -qq install -y openssh-server > /dev/null")
ssh_dir = pathlib.Path("/root/.ssh"); ssh_dir.mkdir(mode=0o700, exist_ok=True)
(ssh_dir / "authorized_keys").write_text(pub + "\n"); os.chmod(ssh_dir / "authorized_keys", 0o600)
pathlib.Path("/etc/ssh/sshd_config.d").mkdir(exist_ok=True)
pathlib.Path("/etc/ssh/sshd_config.d/kaggle.conf").write_text(
    "PasswordAuthentication no\nKbdInteractiveAuthentication no\nPermitRootLogin prohibit-password\n"
    "PubkeyAuthentication yes\nClientAliveInterval 30\n")
sh("mkdir -p /run/sshd && /usr/sbin/sshd")
sh("curl -fsSL -o /usr/local/bin/cloudflared "
   "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared")
print("sshd running (key-only); cloudflared", sh("cloudflared --version").strip())


In [ ]:
log = open("/kaggle/working/cloudflared.log", "w")
proc = subprocess.Popen(["cloudflared", "tunnel", "--no-autoupdate", "--url", "ssh://localhost:22"],
                        stdout=log, stderr=subprocess.STDOUT)
host = None
for _ in range(60):
    time.sleep(2)
    m = re.search(r"https://([a-z0-9-]+\.trycloudflare\.com)", open("/kaggle/working/cloudflared.log").read())
    if m:
        host = m.group(1)
        break
if not host:
    raise SystemExit("no tunnel hostname after 120 s; see /kaggle/working/cloudflared.log")
print("=" * 70)
print("TUNNEL HOST:", host)
print("On the PC:  bash scripts/edge/kaggle_tunnel_client.sh", host)
print("=" * 70, flush=True)


In [ ]:
# Keeps the session alive while the tunnel is up. Interrupt this cell (or stop the session) to end it.
t0 = time.time()
while proc.poll() is None and time.time() - t0 < 11.5 * 3600:
    time.sleep(60)
    print(f"alive {int((time.time() - t0) / 60)} min", end="\r", flush=True)
print("\ntunnel ended")
